In [1]:
# ─── CELL 1: mount & navigate ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir("/content/drive/MyDrive/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence")
sys.path.insert(0, '.')
print("Working dir:", os.getcwd())

Mounted at /content/drive
Working dir: /content/drive/MyDrive/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence


In [2]:
# ─── CELL 2: install ────────────────────────────────────────────────────
!pip install langchain langchain-community langchain-core \
             langchain-groq langgraph groq \
             faiss-cpu sentence-transformers \
             datasets pandas numpy -q
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Done.


In [6]:
# ─── CELL 3: setup LLM ───────────────────────────────────────────────────
from google.colab import userdata
from langchain_groq import ChatGroq

GROQ_API_KEY = userdata.get('GROQ_API')

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=512
)

# test
response = llm.invoke("Say hello in one sentence.")
print("LLM test:", response.content)

LLM test: Hello, how can I assist you today?


In [7]:
# ─── CELL 4: create Retriever Agent ─────────────────────────────────────
retriever_agent_content = '''# agents/retriever.py
"""
Retriever Agent — specialised agent responsible ONLY for
finding relevant documents from the knowledge base.

Role in multi-agent pipeline:
  Input:  user question
  Output: top retrieved document passages
  Tools:  search_documents
"""
from langchain_groq import ChatGroq
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
import os, sys
sys.path.insert(0, '.')

from tools.retrieval_tool import search_documents

RETRIEVER_PROMPT = PromptTemplate.from_template("""
You are a specialised Document Retrieval Agent.
Your ONLY job is to find relevant documents for the given query.
Search the document knowledge base and return the most relevant passages.
Do NOT answer the question — only retrieve relevant documents.

Tools available:
{tools}

Format:
Question: {input}
Thought: I need to search for relevant documents
Action: search_documents
Action Input: the search query
Observation: retrieved documents
Thought: I have retrieved relevant documents
Final Answer: [paste the retrieved documents here as-is]

Begin!
Question: {input}
Thought: {agent_scratchpad}
""")

def create_retriever_agent(llm):
    tools = [search_documents]
    agent = create_react_agent(llm=llm, tools=tools,
                               prompt=RETRIEVER_PROMPT)
    return AgentExecutor(
        agent=agent, tools=tools,
        verbose=True, max_iterations=3,
        handle_parsing_errors=True,
        early_stopping_method="generate"
    )
'''

with open("agents/retriever.py", "w") as f:
    f.write(retriever_agent_content)
print("retriever.py created.")

retriever.py created.


In [8]:
# ─── CELL 5: create Analyst Agent ───────────────────────────────────────
analyst_agent_content = '''# agents/analyst.py
"""
Analyst Agent — specialised agent responsible for analysing
retrieved content and extracting key facts.

Role in multi-agent pipeline:
  Input:  retrieved document passages + original question
  Output: key facts and insights extracted from documents
  Tools:  summarise_text, calculate
"""
from langchain_groq import ChatGroq
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
import sys
sys.path.insert(0, '.')

from tools.summarise_tool import summarise_text
from tools.calculator_tool import calculate

ANALYST_PROMPT = PromptTemplate.from_template("""
You are a specialised Document Analyst Agent.
Your job is to analyse the retrieved documents and extract
key facts relevant to answering the question.
Use tools to summarise long passages or calculate numerical values.

Tools available:
{tools}

Format:
Question: {input}
Thought: what facts do I need to extract
Action: summarise_text or calculate
Action Input: text to summarise or expression to calculate
Observation: result
Thought: I have extracted the key facts
Final Answer: [list the key facts extracted, bullet points]

Begin!
Question: {input}
Thought: {agent_scratchpad}
""")

def create_analyst_agent(llm):
    tools = [summarise_text, calculate]
    agent = create_react_agent(llm=llm, tools=tools,
                               prompt=ANALYST_PROMPT)
    return AgentExecutor(
        agent=agent, tools=tools,
        verbose=True, max_iterations=3,
        handle_parsing_errors=True,
        early_stopping_method="generate"
    )
'''

with open("agents/analyst.py", "w") as f:
    f.write(analyst_agent_content)
print("analyst.py created.")

analyst.py created.


In [9]:
# ─── CELL 6: create Synthesiser Agent ───────────────────────────────────
synthesiser_agent_content = '''# agents/synthesiser.py
"""
Synthesiser Agent — specialised agent responsible for combining
retrieved documents and analyst insights into a final answer.

Role in multi-agent pipeline:
  Input:  original question + retrieved docs + analyst facts
  Output: final coherent answer grounded in documents
  Tools:  none — pure LLM reasoning
"""
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import sys
sys.path.insert(0, '.')

SYSTEM_PROMPT = """You are a specialised Answer Synthesis Agent.
Your job is to combine retrieved documents and extracted facts
into a clear, accurate, concise final answer.

Rules:
- Base your answer ONLY on the provided context
- Be concise — 2-4 sentences maximum
- If context is insufficient, say so clearly
- Do not hallucinate facts not present in the context
"""

def create_synthesiser_agent(llm):
    def synthesise(question: str, retrieved_docs: str,
                   analyst_facts: str) -> str:
        prompt = f"""
Question: {question}

Retrieved Documents:
{retrieved_docs[:1000]}

Analyst Facts:
{analyst_facts[:500]}

Based on the above context, provide a clear and accurate answer
to the question in 2-4 sentences.
"""
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=prompt)
        ]
        response = llm.invoke(messages)
        return response.content.strip()

    return synthesise
'''

with open("agents/synthesiser.py", "w") as f:
    f.write(synthesiser_agent_content)
print("synthesiser.py created.")

synthesiser.py created.


In [16]:
# ─── CELL 7a: rewrite all agents for LangChain 1.3.1 + LangGraph ────────
import os, sys

# ── retriever.py ──────────────────────────────────────────────────────────
retriever = '''# agents/retriever.py
"""
Retriever Agent — finds relevant documents using vector search.
Pure LangGraph node — no legacy AgentExecutor needed.
"""
import sys
sys.path.insert(0, ".")
from tools.retrieval_tool import search_documents

def retriever_node(state: dict, llm) -> dict:
    """Search documents for the given question."""
    print("\\n[RETRIEVER] Searching documents...")
    question  = state["question"]
    retrieved = search_documents.invoke(question)
    print(f"[RETRIEVER] Found {len(retrieved)} chars of content")
    return {**state, "retrieved_docs": retrieved}
'''

# ── analyst.py ────────────────────────────────────────────────────────────
analyst = '''# agents/analyst.py
"""
Analyst Agent — extracts key facts from retrieved documents.
Uses LLM directly to analyse content.
"""
from langchain_core.messages import HumanMessage, SystemMessage

ANALYST_SYSTEM = """You are a Document Analyst.
Extract the key facts from the provided documents that are
relevant to answering the question.
Return a bullet-point list of key facts only.
Be concise — max 5 bullet points."""

def analyst_node(state: dict, llm) -> dict:
    """Extract key facts from retrieved documents."""
    print("\\n[ANALYST] Extracting key facts...")

    prompt = f"""Question: {state["question"]}

Retrieved Documents:
{state["retrieved_docs"][:1500]}

Extract the key facts needed to answer this question."""

    messages = [
        SystemMessage(content=ANALYST_SYSTEM),
        HumanMessage(content=prompt)
    ]
    response = llm.invoke(messages)
    facts    = response.content.strip()
    print(f"[ANALYST] Extracted facts: {facts[:100]}...")
    return {**state, "analyst_facts": facts}
'''

# ── synthesiser.py ────────────────────────────────────────────────────────
synthesiser = '''# agents/synthesiser.py
"""
Synthesiser Agent — combines retrieved docs and facts
into a final coherent answer.
"""
from langchain_core.messages import HumanMessage, SystemMessage

SYNTHESISER_SYSTEM = """You are an Answer Synthesis Agent.
Combine the retrieved documents and extracted facts into
a clear, accurate, concise final answer.
Rules:
- Base answer ONLY on provided context
- Be concise — 2-4 sentences maximum
- If context is insufficient, say so clearly
- Do not hallucinate"""

def synthesiser_node(state: dict, llm) -> dict:
    """Generate final answer from retrieved docs and facts."""
    print("\\n[SYNTHESISER] Generating final answer...")

    prompt = f"""Question: {state["question"]}

Retrieved Documents:
{state["retrieved_docs"][:1000]}

Key Facts:
{state["analyst_facts"][:500]}

Provide a clear and accurate answer in 2-4 sentences."""

    messages = [
        SystemMessage(content=SYNTHESISER_SYSTEM),
        HumanMessage(content=prompt)
    ]
    response     = llm.invoke(messages)
    final_answer = response.content.strip()
    print(f"[SYNTHESISER] Answer: {final_answer[:100]}...")
    return {**state, "final_answer": final_answer}
'''

# write all files
with open("agents/retriever.py",   "w") as f: f.write(retriever)
with open("agents/analyst.py",     "w") as f: f.write(analyst)
with open("agents/synthesiser.py", "w") as f: f.write(synthesiser)
print("All agent files rewritten.")

All agent files rewritten.


In [17]:
# ─── CELL 7b: rewrite graph.py ───────────────────────────────────────────
graph_content = '''# src/graph.py
"""
LangGraph Multi-Agent Orchestration.

Graph structure:
  START → retriever_node → analyst_node → synthesiser_node → END

State accumulates:
  question, retrieved_docs, analyst_facts, final_answer
"""
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import sys
sys.path.insert(0, ".")

from agents.retriever   import retriever_node
from agents.analyst     import analyst_node
from agents.synthesiser import synthesiser_node

# ── State ─────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    question:       str
    retrieved_docs: str
    analyst_facts:  str
    final_answer:   str

# ── Build graph ───────────────────────────────────────────────────────────
def build_graph(llm):
    graph = StateGraph(AgentState)

    graph.add_node("retriever",
                   lambda state: retriever_node(state, llm))
    graph.add_node("analyst",
                   lambda state: analyst_node(state, llm))
    graph.add_node("synthesiser",
                   lambda state: synthesiser_node(state, llm))

    graph.add_edge(START,         "retriever")
    graph.add_edge("retriever",   "analyst")
    graph.add_edge("analyst",     "synthesiser")
    graph.add_edge("synthesiser", END)

    return graph.compile()

def run_pipeline(question: str, llm) -> dict:
    """Run full multi-agent pipeline."""
    pipeline = build_graph(llm)

    initial_state = AgentState(
        question       = question,
        retrieved_docs = "",
        analyst_facts  = "",
        final_answer   = ""
    )

    final_state = pipeline.invoke(initial_state)
    return {
        "question":       question,
        "retrieved_docs": final_state["retrieved_docs"],
        "analyst_facts":  final_state["analyst_facts"],
        "final_answer":   final_state["final_answer"]
    }
'''

with open("src/graph.py", "w") as f:
    f.write(graph_content)
print("graph.py rewritten.")

# clear cache
import importlib
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['agents','tools','src']):
        del sys.modules[mod]

# test import
from src.graph import run_pipeline
print("Import successful.")

graph.py rewritten.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval tool: index built with 4089 vectors
Import successful.


In [18]:
# ─── CELL 8: test full pipeline ──────────────────────────────────────────
from google.colab import userdata
from langchain_groq import ChatGroq
from src.graph import run_pipeline

GROQ_API_KEY = userdata.get('GROQ_API')
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=512
)

# test question
question = "What is the Reserve Bank of Australia and when was it established?"
print(f"\nQUESTION: {question}")
print("="*60)

result = run_pipeline(question, llm)

print("\n" + "="*60)
print("FINAL ANSWER:")
print(result['final_answer'])


QUESTION: What is the Reserve Bank of Australia and when was it established?

[RETRIEVER] Searching documents...
[RETRIEVER] Found 1526 chars of content

[ANALYST] Extracting key facts...
[ANALYST] Extracted facts: Here are the key facts extracted from the documents:

• The Reserve Bank of Australia (RBA) was esta...

[SYNTHESISER] Generating final answer...
[SYNTHESISER] Answer: The Reserve Bank of Australia (RBA) is Australia's central bank and banknote issuing authority. It w...

FINAL ANSWER:
The Reserve Bank of Australia (RBA) is Australia's central bank and banknote issuing authority. It was established on 14 January 1960, when the Reserve Bank Act 1959 removed central banking functions from the Commonwealth Bank. The RBA's assets include the gold and foreign exchange reserves of Australia, estimated to be worth A$101 billion.


In [19]:
# ─── CELL 9: test 2 more questions ───────────────────────────────────────
import time

questions = [
    "What is risk-based authentication?",
    "If a subscription costs $12 per month what is the annual cost?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print('='*60)
    result = run_pipeline(q, llm)
    print(f"FINAL ANSWER: {result['final_answer']}")
    time.sleep(2)


Q: What is risk-based authentication?

[RETRIEVER] Searching documents...
[RETRIEVER] Found 1244 chars of content

[ANALYST] Extracting key facts...
[ANALYST] Extracted facts: Here are the key facts relevant to answering the question "What is risk-based authentication?":

• R...

[SYNTHESISER] Generating final answer...
[SYNTHESISER] Answer: Risk-based authentication (RBA) is a method of applying varying levels of stringency to authenticati...
FINAL ANSWER: Risk-based authentication (RBA) is a method of applying varying levels of stringency to authentication processes based on the likelihood that access to a given system could result in its being compromised. This approach can be categorized as either user-dependent or transaction-dependent, with user-dependent RBA employing the same authentication for every session initiated by a given user. The exact authentication credentials depend on the user, aiming to balance security with user convenience.

Q: If a subscription costs $12 per m

In [21]:
!git config --global user.email "chaitanyamhetre97@email.com"
!git config --global user.name "chaitanyamhetre"

!git add .
!git commit -m "LangGraph multi-agent pipeline working, all 3 agents tested"
!git push
print("Pushed.")

[main c3c64e6] LangGraph multi-agent pipeline working, all 3 agents tested
 6 files changed, 150 insertions(+), 1 deletion(-)
 create mode 100644 agents/analyst.py
 create mode 100644 agents/retriever.py
 create mode 100644 agents/synthesiser.py
 create mode 100644 notebooks/02_agents_and_graph.ipynb
 create mode 100644 src/graph.py
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (11/11), 15.72 KiB | 335.00 KiB/s, done.
Total 11 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/chaitanyamhetre/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence.git
   b452bc4..c3c64e6  main -> main
Pushed.
